# Combining artists line up through out the year

In [11]:
import pandas as pd
from pathlib import Path

# Look for extract folder relative to this notebook
search_paths = [
    Path("../extract"),            # expected from notebook folder
    Path("edc_vegas/extract"),     # if run from repo root
    Path("../data/extract"),       # fallback if files were placed under data/extract
    Path("edc_vegas/data/extract")
]
path = next((p for p in search_paths if p.exists()), None)

if path is None:
    raise FileNotFoundError("No extract directory found. Checked: " + ", ".join(str(p) for p in search_paths))

# Load all per-year CSVs
dfs = [pd.read_csv(file) for file in path.glob("*_edc_artists.csv")]
if len(dfs) == 0:
    raise FileNotFoundError(f"No CSV files found in {path}. Run the collection notebook first.")

# Combine and save
edc_all = pd.concat(dfs, ignore_index=True)
# output_file = path / "edc_all.csv"
# edc_all.to_csv(output_file, index=False)

print(f"Combined {len(edc_all)} rows → {output_file}")
display(edc_all.head())

Combined 6372 rows → ../data/extract/edc_all.csv


,year,artist
0,2026,$ami G
1,2026,2AR
2,2026,33 Below
3,2026,MPH
4,2026,6ejou


# Normalize and strip artist names

In [17]:
# Normalize and strip artist names
artist_cols = [c for c in edc_all.columns if c.lower() == "artist"]
if not artist_cols:
    raise KeyError("No artist column found. Available columns: " + ", ".join(edc_all.columns))

artist_col = artist_cols[0]
edc_all["artist"] = (
    edc_all[artist_col]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace("&", "and")
)

before = len(edc_all)
edc_all = edc_all.drop_duplicates(subset=["year", "artist"]).reset_index(drop=True)
after = len(edc_all)
print(f"Deduped on [year, artist]: {before} → {after}")
display(edc_all.head())

Deduped on [year, artist]: 1857 → 1857


,year,artist
0,2026,$ami g
1,2026,2ar
2,2026,33 below
3,2026,mph
4,2026,6ejou


# Show how many times that artist played at EDC from 2022-2026

In [ ]:
# Count how many times each artist appeared (2022-2026)
if "year" not in edc_all.columns:
    raise KeyError("Column 'year' not found in edc_all. Ensure the combine step added the year column.")

artist_counts = (
    edc_all.groupby("artist").agg(
        total_appearances=("year", "count"),
        years_played=("year", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values("total_appearances", ascending=False)
)

print("Top 10 artists by appearances (2022-2026):")
display(artist_counts.head(10))

# Save outputs
edc_all_file = path / "edc_all.csv"
artist_counts_file = path / "artist_counts_2022_2026.csv"

edc_all.to_csv(edc_all_file, index=False)
artist_counts.to_csv(artist_counts_file, index=False)

print(f"\nSaved combined data to {edc_all_file}")
print(f"Saved appearance counts to {artist_counts_file}")

Top 10 artists by appearances (2022-2026):


,artist,total_appearances,years_played
307,dom dolla,5,"[2022, 2023, 2024, 2025, 2026]"
891,soren,5,"[2022, 2023, 2024, 2025, 2026]"
952,tiësto,5,"[2022, 2023, 2024, 2025, 2026]"
625,marc v,5,"[2022, 2023, 2024, 2025, 2026]"
579,lil texas,5,"[2022, 2023, 2024, 2025, 2026]"
347,excision,5,"[2022, 2023, 2024, 2025, 2026]"
27,adrenalize,5,"[2022, 2023, 2024, 2025, 2026]"
269,deorro,5,"[2022, 2023, 2024, 2025, 2026]"
968,trouble,5,"[2022, 2023, 2024, 2025, 2026]"
647,matroda,5,"[2022, 2023, 2024, 2025, 2026]"



Saved combined data to ../data/extract/edc_all.csv
Saved appearance counts to ../data/extract/artist_counts_2022_2026.csv
